# **Deep Natural Language Processing @ PoliTO**

---


**Teaching Assistant:** Giuseppe Gallipoli

**Credits:** Moreno La Quatra

**Practice 3:** Named Entity Recognition (part 1) & Intent Detection (part 2)

## Named Entity Recognition (NER)

The Named Entity Recognition task aims at identifying and classifying named entities in a text. Named entities are real-world objects such as persons, locations, organizations, etc. The task takes as input a sentence and determines the boundaries of the named entities and their type.

For example, given the sentence:

```
I went to Paris last week.
```

the task is to identify the named entity `Paris` as a location.

Hereafter an illustration of the NER task:

![https://miro.medium.com/max/875/0*mlwDqNm7DFc_4maP.jpeg](https://miro.medium.com/max/875/0*mlwDqNm7DFc_4maP.jpeg)   

In the first part of this practice, you will:
- explore the NER task using pre-trained models available on spaCy and HuggingFace
- evaluate the performance of a spaCy NER model on a custom dataset
- evaluate the performance of a HuggingFace NER model on a custom dataset

NB: The library used to evaluate the performance of the models is `seqeval`, which is a library for evaluating sequence labeling tasks, or `eval4ner` for the NER task.

### **Question 1: Data preparation**

The first step is to prepare the data. In this practice, you will use the WikiGold dataset [1][2], which is a collection of annotated sentences from Wikipedia. The dataset is available in [CONLL](https://simpletransformers.ai/docs/ner-data-formats/#text-file-in-conll-format) format. The dataset is available [here](https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P4/NER/wikigold.conll.txt).

**Please, read carefully the following instructions before starting to work on the practice.**

You need to extract clean sentences (no annotation) and, for each sentence, the corresponding annotations. The dataset has the following format:

- `sentences`: list of sentences
- `annotations`: list of list of entities (both string and class information). E.g., `[[('010', 'MISC'), ('Japanese', 'MISC'), ('The Mad Capsule Markets', 'ORG')], [('Osc-Dis', 'MISC'), ('Introduction 010', 'MISC'), ('Come', 'MISC')], ...]`. You can remove I- prefix because the data collection does not actually contain valuable prefixes.

---


[1] Balasuriya, Dominic, et al. "Named entity recognition in wikipedia."
    Proceedings of the 2009 Workshop on The People's Web Meets NLP: Collaboratively Constructed Semantic Resources. Association for Computational Linguistics, 2009.

[2] Nothman, Joel, et al. "Learning multilingual named entity recognition
    from Wikipedia." Artificial Intelligence 194 (2013): 151-175

---

The following cell downloads the dataset on Google Colab.

In [1]:
!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P4/NER/wikigold.conll.txt

--2025-10-24 11:43:55--  https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P4/NER/wikigold.conll.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 318530 (311K) [text/plain]
Saving to: ‘wikigold.conll.txt’

wikigold.conll.txt  100%[===================>] 311.06K  --.-KB/s    in 0.003s  

2025-10-24 11:43:55 (119 MB/s) - ‘wikigold.conll.txt’ saved [318530/318530]



To avoid spending too much time on data processing, the following cells prepare the dataset for you.
After running the cell, you will have the following variables:
- `sentences_with_labels`: a list of tokens with their corresponding labels
- `sentences`: a list of the sentences in the dataset
- `labels`: a list of lists of labels. Each element in the outer list corresponds to a list of labels for a sentence in the dataset.

Hereafter an example of the provided data:

```python
sentences_with_labels[0] = [
    ['010', 'I-MISC'],
    ['is', 'O'],
    ['the', 'O'],
    ['tenth', 'O'],
    ['album', 'O'],
    ['from', 'O'],
    ['Japanese', 'I-MISC'],
    ['Punk', 'O'],
    ['Techno', 'O'],
    ['band', 'O'],
    ['The', 'I-ORG'],
    ['Mad', 'I-ORG'],
    ['Capsule', 'I-ORG'],
    ['Markets', 'I-ORG'],
    ['.', 'O']
]

sentences[0] = [
    '010 is the tenth album from Japanese Punk Techno band The Mad Capsule Markets .'
]

labels[0] = [
    ('010', 'MISC'),
    ('Japanese', 'MISC'),
    ('The Mad Capsule Markets', 'ORG')
]
```

Please, note that the labels are not in IOB format. You can ignore the I- prefix because the data collection does not actually contain valuable prefixes.
Get familar with the data by printing the first 10 sentences and their corresponding labels. Which are the labels in the dataset?

In [2]:
%%capture
! pip install datasets
! pip install transformers
! pip install spacy
! python -m spacy download en_core_web_sm

In [3]:
def split_text_label(filename):
    f = open(filename)
    split_labeled_text = []
    sentence = []
    for line in f:
        if len(line)==0 or line.startswith('-DOCSTART') or line[0]=="\n":
             if len(sentence) > 0:
                 split_labeled_text.append(sentence)
                 sentence = []
             continue
        splits = line.split(' ')
        sentence.append([splits[0],splits[-1].rstrip("\n")])
    if len(sentence) > 0:
        split_labeled_text.append(sentence)
        sentence = []
    return split_labeled_text
sentences_with_labels = split_text_label("wikigold.conll.txt")

In [4]:
print(sentences_with_labels[0])

[['010', 'I-MISC'], ['is', 'O'], ['the', 'O'], ['tenth', 'O'], ['album', 'O'], ['from', 'O'], ['Japanese', 'I-MISC'], ['Punk', 'O'], ['Techno', 'O'], ['band', 'O'], ['The', 'I-ORG'], ['Mad', 'I-ORG'], ['Capsule', 'I-ORG'], ['Markets', 'I-ORG'], ['.', 'O']]


In [5]:
%%capture
! pip install datasets
! pip install transformers
! pip install spacy
! python -m spacy download en_core_web_sm

In [6]:
sentences = []

for sent_list in sentences_with_labels:
    sentence = [s[0] for s in sent_list]
    sentence = " ".join(sentence)
    sentences.append(sentence)

In [7]:
print(sentences[0])

010 is the tenth album from Japanese Punk Techno band The Mad Capsule Markets .


In [8]:
labels = []
overall_labels = []
for sent_list in sentences_with_labels:
    current_labels = []
    prev = "O"
    current_entity = ""
    for w, l in sent_list:
        overall_labels.append(l)
        if l != "O" and prev != "O":
            if l == prev:
                # continue entity
                current_entity += w + " "
            else:
                # end prev and start a new one
                current_labels.append((current_entity.strip(), prev.split("-")[1]))
                current_entity = w + " "
        elif l == "O" and  prev != "O":
            # end prev
            current_labels.append((current_entity.strip(), prev.split("-")[1]))
            current_entity = ""
        elif l != "O" and prev == "O":
            # start new
            current_entity = w + " "

        prev = l
    labels.append(current_labels)

print (labels)
overall_labels = list(set(overall_labels))
overall_labels = [o for o in overall_labels if o != "O"]
overall_labels = [o.split("-")[1] for o in overall_labels]
print (overall_labels)

[[('010', 'MISC'), ('Japanese', 'MISC'), ('The Mad Capsule Markets', 'ORG')], [('Osc-Dis', 'MISC'), ('Introduction 010', 'MISC'), ('Come', 'MISC')], [('Kojima Minoru', 'PER'), ('Good Day', 'MISC'), ('Wardanceis', 'MISC'), ('UK', 'LOC'), ('Killing Joke', 'ORG')], [('XXX can of This', 'MISC')], [('Cannabis', 'MISC'), ('Cannabis', 'MISC'), ('P.O.P', 'MISC'), ('HUMANITY', 'MISC')], [('UK', 'LOC'), ('OSC-DIS', 'MISC')], [('139th', 'ORG'), ('Camp Howe', 'LOC'), ('Pittsburgh', 'LOC')], [('Frederick H. Collier', 'PER')], [('Second Battle of Bull Run', 'MISC'), ("Howe 's Brigade", 'ORG'), ("Couch 's Division", 'ORG'), ('IV Corps', 'ORG'), ('Army of the Potomac', 'ORG'), ("De Trobriand 's 55th New York", 'ORG'), ('Gardes Lafayette', 'ORG')], [('62d NY', 'ORG'), ('93d PVI', 'ORG'), ('98th PVI', 'ORG'), ('102d PVI', 'ORG')], [("Couch 's Division", 'ORG'), ('Army', 'ORG'), ('Potomac', 'LOC')], [('139th', 'ORG'), ('Poolesville', 'LOC'), ('Sandy Hook', 'LOC'), ('Maryland', 'LOC'), ('Battle of Antieta

In [9]:
print (labels[0])

[('010', 'MISC'), ('Japanese', 'MISC'), ('The Mad Capsule Markets', 'ORG')]


### **Question 2: Inference with spaCy for entity recognition**

spaCy is a free, open-source library for advanced Natural Language Processing in Python. It features NER models for different languages including English.
The models are available [here](https://spacy.io/models).

For this question you are asked to instantiate a spaCy model for English and perform inference on the sentences in the dataset. The English model contains a superset of the labels in the dataset. For this reason, you need to map the labels that are not in the dataset to the `MISC` label.

You are expected to generate an output similar to the following:
```python
[('010', 'MISC'), ('Japanese', 'MISC'), ('The Mad Capsule Markets', 'ORG')]
```

Please pay attention to the token attributes (you can find more information [here](https://spacy.io/api/token#attributes)) and the entity attributes (you can find more information [here](https://spacy.io/api/entityrecognizer)).

The following cell instantiates a spaCy model for English.

In [10]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [16]:
# your code here
predicted_labels_spacy = []
# Get the list of labels from the dataset and handle potential errors during split
allowed_labels = [o.split("-")[1] if "-" in o else o for o in overall_labels]

for sentence in sentences:
    doc = nlp(sentence)
    sentence_entities = []
    for ent in doc.ents:
        # Map labels not in the original dataset to 'MISC'
        label_type = ent.label_ if ent.label_ in allowed_labels else 'MISC'
        sentence_entities.append((ent.text, label_type))
    predicted_labels_spacy.append(sentence_entities)

# Print the first 10 predictions to match the expected output format
for i in range(min(10, len(predicted_labels_spacy))):
    print(predicted_labels_spacy[i])

[('010', 'MISC'), ('tenth', 'MISC'), ('Japanese', 'MISC'), ('The Mad Capsule Markets', 'MISC')]
[('Osc-Dis', 'ORG'), ('Introduction 010', 'MISC')]
[('Kojima Minoru', 'MISC'), ('Good Day', 'MISC'), ('Wardanceis', 'ORG'), ('UK', 'MISC'), ('Killing Joke', 'MISC')]
[('XXX', 'MISC')]
[('Cannabis', 'MISC'), ('Cannabis', 'ORG'), ('P.O.P', 'ORG'), ('HUMANITY', 'MISC')]
[('UK Edition', 'MISC')]
[('139th', 'ORG'), ('Camp Howe', 'MISC'), ('Pittsburgh', 'MISC'), ('September 1 , 1862', 'MISC')]
[('Frederick H. Collier', 'MISC'), ('first', 'MISC')]
[('Second', 'MISC'), ("Howe 's Brigade of Couch 's Division of the IV Corps", 'ORG'), ('Potomac', 'LOC'), ("De Trobriand 's", 'MISC'), ('55th New York', 'MISC'), ('Gardes Lafayette', 'MISC'), ('September 11 , 1862', 'MISC')]
[('62d', 'MISC'), ('93d', 'MISC'), ('PVI', 'ORG'), ('PVI', 'ORG'), ('the 102d PVI', 'ORG')]


### **Question 3: Compute metrics for evaluating NER**

The output of NER models consists of a set of named entities. To evaluate the performance of a model, we need to compare the predicted named entities with the ground truth.

For this question, you need to use [`eval4ner`](https://github.com/cyk1337/eval4ner) package to evaluate the performance of the model.

In [17]:
%%capture
! pip install eval4ner

In [25]:
# your code here
from eval4ner import muc

# Evaluate the spaCy model
results = muc.evaluate_all(labels, predicted_labels_spacy, sentences)

# Print the evaluation results
print(results)


 NER evaluation scores:
  strict mode, Precision=0.2336, Recall=0.1575, F1:0.1796
   exact mode, Precision=0.3915, Recall=0.2367, F1:0.2817
 partial mode, Precision=0.3915, Recall=0.2367, F1:0.2817
    type mode, Precision=0.2336, Recall=0.1575, F1:0.1796
{'strict': {'precision': 0.23359093638279935, 'recall': 0.1575401882215901, 'f1_score': 0.17962739527056956, 'count': 1696}, 'exact': {'precision': 0.39146937771348134, 'recall': 0.23674393627214124, 'f1_score': 0.2816962946630353, 'count': 1696}, 'partial': {'precision': 0.39146937771348134, 'recall': 0.23674393627214124, 'f1_score': 0.2816962946630353, 'count': 1696}, 'type': {'precision': 0.23359093638279935, 'recall': 0.1575401882215901, 'f1_score': 0.17962739527056956, 'count': 1696}}


### **Question 4: Inference with transformers pipeline**

Transformer-based models can be fine-tuned for token-level classification. The task is to classify each token in a sentence and assign it to a class.
The NER task is a token-level classification task and the models can be used for performing inference on the sentences in the dataset.

You can use the pipeline available on the HuggingFace [transformers library](https://huggingface.co/docs/transformers/main_classes/pipelines). The pipeline allows to perform inference on a list of sentences.

Evaluate the **standard** model using the pipeline (`pipe = pipeline("ner")`). Check the documentation here: https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.TokenClassificationPipeline


A few notes about the question (**read carefully**):
1. The output of the pipeline differs with respect to spaCy. Please be sure to process data correctly before running evaluation.
2. `ignore_labels` parameter could be used to exclude labels from the prediction.
3. `##` symbol is used when a token is a continuation of a previous one (Poli + ##TO). You may need to check this specific case to merge the tokens correctly.
4. Use `eval4ner` to evaluate the performance of the model.

In [26]:
%%capture
! pip install transformers
! pip install datasets

In [29]:
from transformers import pipeline
from eval4ner import muc

# Instantiate the NER pipeline
ner_pipeline = pipeline("ner", aggregation_strategy="simple") # Use aggregation_strategy="simple" to merge consecutive tokens of the same entity

# Perform inference on the sentences
predicted_tokens_hf = ner_pipeline(sentences)

# Process the output to get entities in the required format for eval4ner
predicted_labels_hf = []
for sentence_predictions in predicted_tokens_hf:
    sentence_entities = []
    for entity in sentence_predictions:
        # Exclude entities with the label 'O'
        if entity['entity_group'] != 'O':
            sentence_entities.append((entity['word'].replace(" ##", ""), entity['entity_group'])) # Remove " ##" and format as (entity_text, entity_type)
    predicted_labels_hf.append(sentence_entities)

# Evaluate the HuggingFace model using eval4ner
results_hf = muc.evaluate_all(labels, predicted_labels_hf, sentences)

# Print the evaluation results
print("HuggingFace Model Evaluation:")
print(results_hf)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0



 NER evaluation scores:
  strict mode, Precision=0.7831, Recall=0.4788, F1:0.5772
   exact mode, Precision=0.8893, Recall=0.5259, F1:0.6414
 partial mode, Precision=0.8893, Recall=0.5259, F1:0.6414
    type mode, Precision=0.7831, Recall=0.4788, F1:0.5772
HuggingFace Model Evaluation:
{'strict': {'precision': 0.7830666987624526, 'recall': 0.47882377000966775, 'f1_score': 0.5772040425549232, 'count': 1696}, 'exact': {'precision': 0.8892875582881477, 'recall': 0.5258793319171876, 'f1_score': 0.6414387006216926, 'count': 1696}, 'partial': {'precision': 0.8892875582881477, 'recall': 0.5258793319171876, 'f1_score': 0.6414387006216926, 'count': 1696}, 'type': {'precision': 0.7830666987624526, 'recall': 0.47882377000966775, 'f1_score': 0.5772040425549232, 'count': 1696}}
